# 04 — Desbalanceamento e limiar

## Objetivo

O desbalanceamento deve ser tratado por pesos, SMOTENC ou por uma decisão de
limiar?

Comparamos somente as três estratégias já validadas. Nenhum novo resampler é
adicionado.

In [ ]:
from pathlib import Path
import sys

ponto_atual = Path.cwd().resolve()
RAIZ = next(
    caminho for caminho in (ponto_atual, *ponto_atual.parents)
    if (caminho / "data" / "raw" / "UCI_Credit_Card.csv").exists()
)
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))


import json
import time
import pandas as pd
from imblearn.over_sampling import SMOTENC
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.utils.class_weight import compute_sample_weight

from src.auxiliares import (
    COLUNAS_NOMINAIS,
    COLUNAS_STATUS,
)
from src.visual_utils import grafico_metricas_por_limiar

pasta_processados = RAIZ / "data" / "processed"
dados_treino = pd.read_csv(pasta_processados / "treino.csv")
dados_validacao = pd.read_csv(pasta_processados / "validacao.csv")

X_treino = dados_treino.drop(columns=["inadimplente"])
y_treino = dados_treino["inadimplente"]

X_validacao = dados_validacao.drop(columns=["inadimplente"])
y_validacao = dados_validacao["inadimplente"]

caminho_parametros = RAIZ / "models" / "parametros_gradient_boosting.json"
if not caminho_parametros.exists():
    raise FileNotFoundError(
        "Execute primeiro o notebook 03_otimizacao.ipynb."
    )

parametros = json.loads(caminho_parametros.read_text(encoding="utf-8"))

## 4.1 — Como se comporta o modelo original?

## 4.2 — Pesos aumentam o alcance da classe positiva?

## 4.3 — O SMOTENC melhora o ranking?

Os códigos nominais e de status de pagamento são categorias do domínio. Para
evitar sua interpolação como valores contínuos, usamos
`COLUNAS_NOMINAIS + COLUNAS_STATUS` como features categóricas do SMOTENC.

## 4.4 — Qual estratégia preserva melhor a Average Precision (AP)?

In [ ]:
colunas_comparacao = [
    "modelo",
    "average_precision",
    "precision",
    "recall",
    "f1",
    "fp",
]

resultados_balanceamento[
    colunas_comparacao
].sort_values("average_precision", ascending=False)

A comparação considera o ranking probabilístico pela Average Precision e
também o efeito em Precision, Recall e falsos positivos. Pesos elevam Recall,
mas aumentam os falsos positivos; o SMOTENC adiciona complexidade sem melhorar
o ranking neste experimento. Por isso, seguimos com os dados originais.

## 4.5 — O que muda quando alteramos o limiar?

In [ ]:
fig = grafico_metricas_por_limiar(resultados_limiares)
fig.show()

## 4.6 — Resultado

O modelo com dados originais permanece como solução final. Na validação,
avaliamos diferentes limiares e observamos o trade-off entre Precision e
Recall. Para este projeto didático seguimos com 0,27, decisão tomada antes de
abrir o teste. Esse valor não é universal e depende do contexto de uso.